In [1]:
import pandas as pd

df = pd.read_csv("money_company.csv")

print(df)

     일련번호                           상품명      중량     판매가         소재  \
0       1   2023 계묘년 토끼의 해 카드형 골드 1.87g    1.87   310000  Au 999.9   
1       2  2023 계묘년 토끼의 해 카드형 골드 11.25g   11.25  1760000  Au 999.9   
2       3   2023 계묘년 토끼의 해 카드형 골드 3.75g    3.75   600000  Au 999.9   
3       4   2023 계묘년 토끼의 해 카드형 골드 37.5g   37.50  5900000  Au 999.9   
4       5    2024 갑진년 용의 해 카드형 골드 1.87g    1.87   310000  Au 999.9   
..    ...                           ...     ...      ...       ...   
120   121         로열시리즈 4차-일월오봉도 기념 은메달   10.00   110000    Ag 999   
121   122                  초콜릿 실버바_핑크베리    0.00   297000    Ag9999   
122   123                    초콜릿 실버바_밀크  100.00   264000  Ag 999.9   
123   124                   초콜릿 실버바_화이트  100.00   231000  Ag 999.9   
124   125              가화만사성 실버바(93.3g)   93.30   220000  Ag 999.9   

                                 규격                      구성  \
0            골드바 12x20mm 카드 86x54mm    카드형 골드, 보증서, 아우터 케이스   
1            골드바 16x27mm 카드 86x54

In [11]:
import pandas as pd
import sqlite3

# CSV 로드
df = pd.read_csv("money_company.csv")

# DB 연결
conn = sqlite3.connect("shop.db")

# 테이블 저장
df.to_sql("products", conn, if_exists="replace", index=False)

# 인덱스 생성 (검색 성능 향상)
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_product_name ON products(상품명)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_material ON products(소재)")

conn.commit()
conn.close()

print(f"✅ {len(df)}개 상품을 shop.db에 저장했습니다.")

✅ 125개 상품을 shop.db에 저장했습니다.


In [18]:
import pandas as pd
import sqlite3

# SQLite3로 shop.db 연결
conn = sqlite3.connect("shop.db")

# 컬럼명의 공백 문제 해결
try:
    # 상품명, 중량, 판매가, 소재만 선택 (공백 제거)
    df = pd.read_sql("""
        SELECT 상품명, 중량, [ 판매가 ] as 판매가, 소재 
        FROM products 
        WHERE [ 판매가 ] IS NOT NULL 
        AND CAST([ 판매가 ] AS INTEGER) > 0
        ORDER BY CAST([ 판매가 ] AS INTEGER) DESC
        LIMIT 10
    """, conn)
    
    print("상품명, 중량, 판매가, 소재만:")
    print(df)
    
    # 유효한 상품 수
    valid_count = pd.read_sql("""
        SELECT COUNT(*) as count 
        FROM products 
        WHERE [ 판매가 ] IS NOT NULL 
        AND CAST([ 판매가 ] AS INTEGER) > 0
    """, conn)
    
    print(f"\n판매가 있는 상품 수: {valid_count.iloc[0]['count']}개")
    
except Exception as e:
    print(f"오류 발생: {e}")

conn.close()

상품명, 중량, 판매가, 소재만:
                                    상품명    중량      판매가        소재
0            2025 을사년 뱀의 해 카드형 골드 37.5g  37.5  5960000  Au 999.9
1           2023 계묘년 토끼의 해 카드형 골드 37.5g  37.5  5900000  Au 999.9
2            2024 갑진년 용의 해 카드형 골드 37.5g  37.5  5900000  Au 999.9
3                   가화만사성 카드형 골드(37.5g)  37.5  5900000   Au999.9
4       국보 반가사유상(옛 국보 78호) 카드형 골드 37.5g  37.5  5900000  Au 999.9
5       국보 반가사유상(옛 국보 83호) 카드형 골드 37.5g  37.5  5900000  Au 999.9
6          로열시리즈 2차 해학반도도 카드형 골드 37.5g   37.5  5900000    Au9999
7                    일월오봉도 카드형 골드 37.5g  37.5  5900000  Au 999.9
8                   천마총 금관 카드형 골드 37.5g  37.5  5900000  Au 999.9
9  한국의 대표 화가 3차 - 김환기 기념메달 금메달(매화와 항아리)  31.1  5400000    Au 999

판매가 있는 상품 수: 125개


In [9]:
import sqlite3
import pandas as pd

# users 테이블 생성
conn = sqlite3.connect("shop.db")
cursor = conn.cursor()

# users 테이블 생성
cursor.execute('''
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT UNIQUE NOT NULL,
        name TEXT NOT NULL,
        password TEXT NOT NULL,
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    )
''')

# 샘플 사용자 데이터 삽입 (테스트용)
sample_users = [
    ('admin@subbot.com', '관리자', 'hashed_password_123', '2024-01-01 00:00:00'),
    ('user@test.com', '테스트유저', 'hashed_password_456', '2024-01-02 00:00:00')
]

cursor.executemany('''
    INSERT OR IGNORE INTO users (email, name, password, created_at) 
    VALUES (?, ?, ?, ?)
''', sample_users)

conn.commit()

# 확인
df = pd.read_sql("SELECT * FROM users", conn)
print("users 테이블 생성 완료:")
print(df)

conn.close()

users 테이블 생성 완료:
   id             email   name             password           created_at
0   1  admin@subbot.com    관리자  hashed_password_123  2024-01-01 00:00:00
1   2     user@test.com  테스트유저  hashed_password_456  2024-01-02 00:00:00


In [10]:
# SQLAlchemy 설치 및 사용 방법
try:
    from sqlalchemy import create_engine, text
    import pandas as pd
    
    # 엔진 생성
    engine = create_engine("sqlite:///shop.db")
    
    # 쿼리 실행
    with engine.connect() as conn:
        df = pd.read_sql("SELECT * FROM products LIMIT 5", conn)
        print("SQLAlchemy로 products 데이터 조회:")
        print(df)
        
except ImportError:
    print("SQLAlchemy가 설치되지 않았습니다. 아래 명령어로 설치하세요:")
    print("!pip install sqlalchemy")
    
except Exception as e:
    print(f"오류 발생: {e}")
    print("sqlite3를 직접 사용하는 것을 권장합니다.")

오류 발생: Class <class 'sqlalchemy.sql.elements.SQLCoreOperations'> directly inherits TypingOnly but has additional attributes {'__static_attributes__', '__firstlineno__'}.
sqlite3를 직접 사용하는 것을 권장합니다.


In [ ]:
import sqlite3
import pandas as pd

class ProductChatBot:
    def __init__(self):
        # shop.db 연결
        self.conn = sqlite3.connect("shop.db")
        
    def get_product_info(self, product_name):
        """상품명으로 관련 정보 검색"""
        try:
            # 단순화된 검색: 키워드 분리
            keywords = product_name.split()
            
            # 각 키워드로 개별 검색
            all_results = []
            for keyword in keywords:
                if len(keyword.strip()) > 0:
                    query = """
                        SELECT 상품명, 중량, [ 판매가 ] as 판매가, 소재 
                        FROM products 
                        WHERE 상품명 LIKE ? 
                        AND [ 판매가 ] IS NOT NULL 
                        AND CAST([ 판매가 ] AS INTEGER) > 0
                        ORDER BY CAST([ 판매가 ] AS INTEGER) DESC
                        LIMIT 3
                    """
                    
                    df = pd.read_sql(query, self.conn, params=[f"%{keyword.strip()}%"])
                    if len(df) > 0:
                        all_results.extend(df.to_dict('records'))
            
            # 중복 제거
            unique_results = []
            seen_names = set()
            for item in all_results:
                if item['상품명'] not in seen_names:
                    unique_results.append(item)
                    seen_names.add(item['상품명'])
            
            if unique_results:
                df_result = pd.DataFrame(unique_results[:5])  # 상위 5개만
                return self._format_product_response(df_result, product_name)
            else:
                return self._no_product_response(product_name)
                
        except Exception as e:
            return f"오류 발생: {e}"
    
    def _format_product_response(self, df, search_term):
        """상품 정보 응답 포맷팅"""
        response = f"🔍 '{search_term}' 관련 상품 정보입니다:\n\n"
        
        for idx, row in df.iterrows():
            response += f"📦 {idx+1}. {row['상품명']}\n"
            response += f"   💰 가격: {row['판매가']:,}원\n"
            response += f"   ⚖️  중량: {row['중량']}g\n"
            response += f"   🏷️  소재: {row['소재']}\n\n"
        
        response += f"💡 더 자세한 정보는 문의해주세요.\n"
        response += f"📞 고객센터: 1577-4321"
        
        return response
    
    def _no_product_response(self, product_name):
        """상품이 없을 때 응답"""
        return f"""❌ '{product_name}'에 대한 상품을 찾을 수 없습니다.

💡 검색 팁:
• 정확한 상품명 입력 (예: '토끼 골드', '은메달')
• 연도 포함 검색 (예: '2023 골드', '2024 메달')
• 소재로 검색 (예: '금바', '은메달', '골드바')
• 키워드 단축 검색 (예: '토끼', '골드', '메달')

📞 직접 문의: 1577-4321
📧 이메일: support@subbot.com"""

# 챗봇 테스트
bot = ProductChatBot()

# 테스트 질문들
test_questions = [
    "토끼 골드",
    "은메달 가격", 
    "2023 금바",
    "카드형 골드",
    "금",
    "은"
]

print("🤖 상품 정보 챗봇 테스트\n" + "="*50)

for question in test_questions:
    print(f"\n👤 사용자: {question}")
    response = bot.get_product_info(question)
    print(f"🤖 챗봇: {response}")
    print("-" * 50)

🤖 상품 정보 챗봇 테스트

👤 사용자: 토끼 골드
🤖 챗봇: ❌ '토끼 골드'에 대한 상품을 찾을 수 없습니다.

💡 검색 팁:
• 정확한 상품명 입력 (예: '토끼 골드', '은메달')
• 연도 포함 검색 (예: '2023 골드', '2024 메달')
• 소재로 검색 (예: '금바', '은메달', '골드바')
• 키워드 단축 검색 (예: '토끼', '골드', '메달')

📞 직접 문의: 1577-4321
📧 이메일: support@subbot.com
--------------------------------------------------

👤 사용자: 은메달 가격
🤖 챗봇: ❌ '은메달 가격'에 대한 상품을 찾을 수 없습니다.

💡 검색 팁:
• 정확한 상품명 입력 (예: '토끼 골드', '은메달')
• 연도 포함 검색 (예: '2023 골드', '2024 메달')
• 소재로 검색 (예: '금바', '은메달', '골드바')
• 키워드 단축 검색 (예: '토끼', '골드', '메달')

📞 직접 문의: 1577-4321
📧 이메일: support@subbot.com
--------------------------------------------------

👤 사용자: 2023 금바
🤖 챗봇: ❌ '2023 금바'에 대한 상품을 찾을 수 없습니다.

💡 검색 팁:
• 정확한 상품명 입력 (예: '토끼 골드', '은메달')
• 연도 포함 검색 (예: '2023 골드', '2024 메달')
• 소재로 검색 (예: '금바', '은메달', '골드바')
• 키워드 단축 검색 (예: '토끼', '골드', '메달')

📞 직접 문의: 1577-4321
📧 이메일: support@subbot.com
--------------------------------------------------

👤 사용자: 카드형 골드
🤖 챗봇: 🔍 '카드형 골드' 관련 상품 정보입니다:

📦 1. 2025 을사년 뱀의 해 카드형 골드 37.5g
   💰 가격: 5,960,000원
   ⚖️

In [22]:
import sqlite3
import pandas as pd

# 데이터베이스 상태 확인
conn = sqlite3.connect("shop.db")

try:
    # 1. 테이블 목록 확인
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = cursor.fetchall()
    print("📋 데이터베이스 테이블 목록:")
    for table in tables:
        print(f"  - {table[0]}")
    
    print("\n" + "="*50)
    
    # 2. products 테이블 데이터 수 확인
    cursor.execute("SELECT COUNT(*) FROM products")
    total_count = cursor.fetchone()[0]
    print(f"📦 products 테이블 총 데이터 수: {total_count}개")
    
    # 3. 실제 데이터 샘플 확인
    if total_count > 0:
        df_sample = pd.read_sql("SELECT * FROM products LIMIT 3", conn)
        print("\n📋 products 테이블 샘플 데이터:")
        print(df_sample.to_string())
    else:
        print("\n❌ products 테이블에 데이터가 없습니다!")
    
    # 4. 컬럼 정보 확인
    cursor.execute("PRAGMA table_info(products)")
    columns = cursor.fetchall()
    print(f"\n📋 products 테이블 컬럼 정보:")
    for col in columns:
        print(f"  - {col[1]} ({col[2]})")
        
except Exception as e:
    print(f"❌ 데이터베이스 확인 오류: {e}")

finally:
    conn.close()

📋 데이터베이스 테이블 목록:
  - users
  - sqlite_sequence
  - products

📦 products 테이블 총 데이터 수: 125개

📋 products 테이블 샘플 데이터:
   일련번호                           상품명     중량     판매가         소재                              규격                    구성                                                                                                                     상품정보고시
0     1   2023 계묘년 토끼의 해 카드형 골드 1.87g   1.87   310000  Au 999.9          골드바 12x20mm 카드 86x54mm  카드형 골드, 보증서, 아우터 케이스  소재:순금 Au999.9크기:(가로X세로) 골드바 12x20mm 카드 86x54mm무게:1.87g구성:카드형 골드, 보증서, 아우터 케이스보증서:품질을 보증하는 보증서 제공품질보증:한국조폐공사상품문의:1577-4321
1     2  2023 계묘년 토끼의 해 카드형 골드 11.25g  11.25  1760000  Au 999.9          골드바 16x27mm 카드 86x54mm  카드형 골드, 보증서, 아우터 케이스         소재:순금 Au999.9크기:골드바 16x27mm 카드 86x54mm무게:11.25g구성:카드형 골드, 보증서, 아우터 케이스보증서:품질을 보증하는 보증서 제공품질보증:한국조폐공사상품문의:1577-4321
2     3   2023 계묘년 토끼의 해 카드형 골드 3.75g   3.75   600000  Au 999.9  (가로X세로) 골드바 12X20mm 카드 86X54mm  카드형 골드, 보증서, 아우터 케이스  소재:순금 Au999.9크기:(가로X세로) 골드바 12X20mm 카드 86X54

In [25]:
import sqlite3
import pandas as pd

# 직접 데이터 확인 및 테스트
conn = sqlite3.connect("shop.db")

try:
    # 1. '토끼'가 포함된 상품명 직접 확인
    cursor = conn.cursor()
    cursor.execute("SELECT 상품명, 중량, [ 판매가 ] as 판매가, 소재 FROM products WHERE 상품명 LIKE '%토끼%' LIMIT 5")
    rabbit_results = cursor.fetchall()
    
    print("🔍 '토끼'가 포함된 상품:")
    for i, row in enumerate(rabbit_results, 1):
        print(f"  {i}. {row[0]} - {row[2]}원")
    
    print("\n" + "="*50)
    
    # 2. '골드'가 포함된 상품명 확인
    cursor.execute("SELECT 상품명, 중량, [ 판매가 ] as 판매가, 소재 FROM products WHERE 상품명 LIKE '%골드%' LIMIT 5")
    gold_results = cursor.fetchall()
    
    print("🔍 '골드'가 포함된 상품:")
    for i, row in enumerate(gold_results, 1):
        print(f"  {i}. {row[0]} - {row[2]}원")
    
    print("\n" + "="*50)
    
    # 3. 모든 상품명의 처음 10개 확인
    cursor.execute("SELECT 상품명 FROM products LIMIT 10")
    all_products = cursor.fetchall()
    
    print("📋 모든 상품명 (처음 10개):")
    for i, row in enumerate(all_products, 1):
        print(f"  {i}. {row[0]}")
        
except Exception as e:
    print(f"❌ 오류: {e}")

finally:
    conn.close()

🔍 '토끼'가 포함된 상품:
  1. 2023 계묘년 토끼의 해 카드형 골드 1.87g - 310000원
  2. 2023 계묘년 토끼의 해 카드형 골드 11.25g - 1760000원
  3. 2023 계묘년 토끼의 해 카드형 골드 3.75g - 600000원
  4. 2023 계묘년 토끼의 해 카드형 골드 37.5g - 5900000원
  5. 디윰아트 2023 계묘년 토끼의 해 고심도 메달 - 499000원

🔍 '골드'가 포함된 상품:
  1. 2023 계묘년 토끼의 해 카드형 골드 1.87g - 310000원
  2. 2023 계묘년 토끼의 해 카드형 골드 11.25g - 1760000원
  3. 2023 계묘년 토끼의 해 카드형 골드 3.75g - 600000원
  4. 2023 계묘년 토끼의 해 카드형 골드 37.5g - 5900000원
  5. 2024 갑진년 용의 해 카드형 골드 1.87g - 310000원

📋 모든 상품명 (처음 10개):
  1. (SWEET MOMENT)라이징 스타
  2. (SWEET MOMENT)블루밍 플라워
  3. (SWEET MOMENT)스위트 하트
  4. 2023 계묘년 토끼의 해 카드형 골드 1.87g
  5. 2023 계묘년 토끼의 해 카드형 골드 11.25g
  6. 2023 계묘년 토끼의 해 카드형 골드 3.75g
  7. 2023 계묘년 토끼의 해 카드형 골드 37.5g
  8. 2024 갑진년 용의 해 카드형 골드 1.87g
  9. 2024 갑진년 용의 해 카드형 골드 11.25g
  10. 2024 갑진년 용의 해 카드형 골드 3.75g


In [26]:
# 완성된 상품 정보 챗봇 코드
# 이 코드를 복사해서 backend 폴더에 product_chatbot.py 파일로 저장하면 됩니다

import sqlite3
import pandas as pd
import re
from typing import Dict, List

class ProductChatBot:
    def __init__(self):
        # shop.db 연결
        self.conn = sqlite3.connect("shop.db")
        
    def get_product_info(self, product_name: str) -> str:
        """상품명으로 관련 정보 검색"""
        try:
            # 키워드 분리 검색
            keywords = product_name.split()
            
            all_results = []
            for keyword in keywords:
                if len(keyword.strip()) > 0:
                    query = """
                        SELECT 상품명, 중량, [ 판매가 ] as 판매가, 소재 
                        FROM products 
                        WHERE 상품명 LIKE ? 
                        AND [ 판매가 ] IS NOT NULL 
                        AND CAST([ 판매가 ] AS INTEGER) > 0
                        ORDER BY CAST([ 판매가 ] AS INTEGER) DESC
                        LIMIT 3
                    """
                    
                    df = pd.read_sql(query, self.conn, params=[f"%{keyword.strip()}%"])
                    if len(df) > 0:
                        all_results.extend(df.to_dict('records'))
            
            # 중복 제거
            unique_results = []
            seen_names = set()
            for item in all_results:
                if item['상품명'] not in seen_names:
                    unique_results.append(item)
                    seen_names.add(item['상품명'])
            
            if unique_results:
                df_result = pd.DataFrame(unique_results[:5])
                return self._format_product_response(df_result, product_name)
            else:
                return self._no_product_response(product_name)
                
        except Exception as e:
            return f"오류 발생: {e}"
    
    def _format_product_response(self, df: pd.DataFrame, search_term: str) -> str:
        """상품 정보 응답 포맷팅"""
        response = f"🔍 '{search_term}' 관련 상품 정보입니다:\n\n"
        
        for idx, row in df.iterrows():
            response += f"📦 {idx+1}. {row['상품명']}\n"
            response += f"   💰 가격: {row['판매가']:,}원\n"
            response += f"   ⚖️  중량: {row['중량']}g\n"
            response += f"   🏷️  소재: {row['소재']}\n\n"
        
        response += f"💡 더 자세한 정보는 문의해주세요.\n"
        response += f"📞 고객센터: 1577-4321"
        
        return response
    
    def _no_product_response(self, product_name: str) -> str:
        """상품이 없을 때 응답"""
        return f"""❌ '{product_name}'에 대한 상품을 찾을 수 없습니다.

💡 검색 팁:
• 정확한 상품명 입력 (예: '토끼 골드', '은메달')
• 연도 포함 검색 (예: '2023 골드', '2024 메달')
• 소재로 검색 (예: '금바', '은메달', '골드바')

📞 직접 문의: 1577-4321
📧 이메일: support@subbot.com"""

# 챗봇 테스트
def test_product_chatbot():
    bot = ProductChatBot()
    
    test_questions = [
        "토끼 골드",
        "은메달 가격", 
        "2023 금바",
        "카드형 골드",
        "금",
        "은"
    ]
    
    print("🤖 상품 정보 챗봇 테스트\n" + "="*50)
    
    for question in test_questions:
        print(f"\n👤 사용자: {question}")
        response = bot.get_product_info(question)
        print(f"🤖 챗봇: {response}")
        print("-" * 50)

# 테스트 실행
if __name__ == "__main__":
    test_product_chatbot()

🤖 상품 정보 챗봇 테스트

👤 사용자: 토끼 골드
🤖 챗봇: 🔍 '토끼 골드' 관련 상품 정보입니다:

📦 1. 2023 계묘년 토끼의 해 카드형 골드 37.5g
   💰 가격: 5,900,000원
   ⚖️  중량: 37.5g
   🏷️  소재: Au 999.9

📦 2. 2023 계묘년 토끼의 해 카드형 골드 11.25g
   💰 가격: 1,760,000원
   ⚖️  중량: 11.25g
   🏷️  소재: Au 999.9

📦 3. 천사의 재능 십이지 메달 12종(토끼) 
   💰 가격: 748,000원
   ⚖️  중량: 4.65g
   🏷️  소재: 메달 Au 999/ 베젤 Au585

📦 4. 2025 을사년 뱀의 해 카드형 골드 37.5g
   💰 가격: 5,960,000원
   ⚖️  중량: 37.5g
   🏷️  소재: Au 999.9

📦 5. 2024 갑진년 용의 해 카드형 골드 37.5g
   💰 가격: 5,900,000원
   ⚖️  중량: 37.5g
   🏷️  소재: Au 999.9

💡 더 자세한 정보는 문의해주세요.
📞 고객센터: 1577-4321
--------------------------------------------------

👤 사용자: 은메달 가격
🤖 챗봇: 🔍 '은메달 가격' 관련 상품 정보입니다:

📦 1. 한반도 공룡 시리즈 1차 코리아노사우루스 보성엔시스 기념 은메달
   💰 가격: 1,188,000원
   ⚖️  중량: 408.0g
   🏷️  소재: Ag999

📦 2. 한국의 대표 화가 3차 - 김환기 기념메달 은메달(매화와 항아리)
   💰 가격: 285,000원
   ⚖️  중량: 31.1g
   🏷️  소재: Ag 999

📦 3. 한국의 대표 화가 3차 - 김환기 기념메달 은메달(영원의 노래)
   💰 가격: 285,000원
   ⚖️  중량: 31.1g
   🏷️  소재: Ag 999

💡 더 자세한 정보는 문의해주세요.
📞 고객센터: 1577-4321
---------------------

In [27]:
# Ollama phi3:latest 테스트 코드
import requests
import json

def test_ollama_phi3():
    """Ollama phi3:latest 모델 테스트"""
    
    # Ollama API 엔드포인트
    url = "http://localhost:11434/api/generate"
    
    # 테스트 프롬프트들
    test_prompts = [
        "상품 '토끼 골드'에 대한 정보를 알려주세요",
        "은메달 가격이 얼마인가요?",
        "2023년 금바 상품 정보를 알려주세요",
        "안녕하세요",
        "오늘 날씨가 어떤가요?"
    ]
    
    print("🤖 Ollama phi3:latest 테스트 시작\n" + "="*60)
    
    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n📝 테스트 {i}: {prompt}")
        print("-" * 40)
        
        # 요청 데이터
        data = {
            "model": "phi3:latest",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.7,
                "max_tokens": 500
            }
        }
        
        try:
            # API 요청
            response = requests.post(url, json=data, timeout=30)
            
            if response.status_code == 200:
                result = response.json()
                answer = result.get("response", "")
                print(f"✅ 응답: {answer}")
                
                # 응답 품질 평가
                if any(keyword in answer.lower() for keyword in ["골드", "메달", "금", "은", "가격", "원"]):
                    print("🎯 상품 관련 응답 감지")
                elif "안녕" in answer or "hello" in answer.lower():
                    print("👋 인사 응답 감지")
                elif "날씨" in answer or "weather" in answer.lower():
                    print("🌤️ 날씨 응답 감지")
                else:
                    print("❓ 기타 응답")
                    
            else:
                print(f"❌ 오류: {response.status_code} - {response.text}")
                
        except requests.exceptions.Timeout:
            print("⏰ 타임아웃 오류 (30초)")
        except requests.exceptions.ConnectionError:
            print("🔌 연결 오류 - Ollama 서버가 실행 중인지 확인하세요")
        except Exception as e:
            print(f"❌ 예외 오류: {str(e)}")
    
    print("\n" + "="*60)
    print("🏁 Ollama phi3:latest 테스트 완료")

def check_ollama_status():
    """Ollama 서버 상태 확인"""
    try:
        # 모델 목록 확인
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json().get("models", [])
            print("📋 사용 가능한 모델:")
            for model in models:
                name = model.get("name", "")
                size = model.get("size", 0)
                print(f"  - {name} ({size/1024/1024/1024:.1f}GB)")
            
            # phi3:latest가 있는지 확인
            phi3_available = any(model.get("name") == "phi3:latest" for model in models)
            if phi3_available:
                print("✅ phi3:latest 모델 사용 가능")
            else:
                print("❌ phi3:latest 모델을 찾을 수 없습니다")
                print("💡 설치 명령어: ollama pull phi3:latest")
                
        else:
            print(f"❌ Ollama 서버 응답 오류: {response.status_code}")
            
    except requests.exceptions.ConnectionError:
        print("❌ Ollama 서버에 연결할 수 없습니다")
        print("💡 Ollama 설치 및 실행:")
        print("   1. https://ollama.ai/ 에서 설치")
        print("   2. 터미널에서 'ollama serve' 실행")
        print("   3. 'ollama pull phi3:latest'로 모델 설치")
    except Exception as e:
        print(f"❌ 상태 확인 오류: {str(e)}")

# 실행
if __name__ == "__main__":
    print("🔍 Ollama 서버 상태 확인...")
    check_ollama_status()
    
    print("\n" + "="*60)
    print("🧪 phi3:latest 모델 테스트 시작...")
    test_ollama_phi3()

🔍 Ollama 서버 상태 확인...
📋 사용 가능한 모델:
  - phi3:latest (2.0GB)
✅ phi3:latest 모델 사용 가능

🧪 phi3:latest 모델 테스트 시작...
🤖 Ollama phi3:latest 테스트 시작

📝 테스트 1: 상품 '토끼 골드'에 대한 정보를 알려주세요
----------------------------------------
⏰ 타임아웃 오류 (30초)

📝 테스트 2: 은메달 가격이 얼마인가요?
----------------------------------------
✅ 응답: 다음 장소에서 은메달의 가격을 알아보도록 하겠습니다. 여기에서 지시한 어느 곳이었나요?
🎯 상품 관련 응답 감지

📝 테스트 3: 2023년 금바 상품 정보를 알려주세요
----------------------------------------
⏰ 타임아웃 오류 (30초)

📝 테스트 4: 안녕하세요
----------------------------------------
✅ 응답: Hello there! How can I help you today?
👋 인사 응답 감지

📝 테스트 5: 오늘 날씨가 어떤가요?
----------------------------------------
⏰ 타임아웃 오류 (30초)

🏁 Ollama phi3:latest 테스트 완료


In [28]:
# shop.db에서 직접 데이터를 불러오는 상품 정보 챗봇 (LLM 없음)
import sqlite3
import pandas as pd
import re
from typing import Dict, List, Optional

class DirectProductChatBot:
    def __init__(self):
        # shop.db 연결
        self.conn = sqlite3.connect("shop.db")
        self._cache = {}  # 검색 결과 캐시
        
    def get_product_info(self, query: str) -> str:
        """상품 관련 질문에 직접 답변"""
        query_lower = query.lower()
        
        # 가격 관련 질문
        if any(keyword in query_lower for keyword in ['가격', '얼마', '비용', '금액', '판매가']):
            return self._get_price_response(query)
        
        # 상품 정보 질문
        elif any(keyword in query_lower for keyword in ['상품', '정보', '알려', '찾아']):
            return self._get_product_response(query)
        
        # 인사
        elif any(keyword in query_lower for keyword in ['안녕', '반가워', '하이', '헬로']):
            return self._get_greeting_response()
        
        # 기타 응답
        else:
            return self._get_default_response(query)
    
    def _get_price_response(self, query: str) -> str:
        """가격 관련 응답"""
        # 상품명 추출
        product_name = self._extract_product_name(query)
        
        if not product_name:
            return """💰 가격 문의

정확한 상품명을 말씀해주세요.

💡 예시:
• '토끼 골드 가격'
• '은메달 얼마'
• '2023 금바 가격'

📞 고객센터: 1577-4321"""
        
        # 상품 검색
        products = self._search_products(product_name)
        
        if not products:
            return f"""❌ '{product_name}' 상품을 찾을 수 없습니다.

💡 검색 팁:
• '토끼 골드', '은메달', '금바' 등으로 검색
• 연도 포함: '2023 골드', '2024 메달'

📞 직접 문의: 1577-4321"""
        
        # 가격 정보 포맷팅
        response = f"💰 '{product_name}' 관련 상품 가격 정보:\n\n"
        
        for i, product in enumerate(products[:5], 1):
            response += f"📦 {i}. {product['상품명']}\n"
            response += f"   💵 가격: {product['판매가']:,}원\n"
            response += f"   ⚖️  중량: {product['중량']}g\n"
            response += f"   🏷️  소재: {product['소재']}\n\n"
        
        response += f"💡 더 자세한 정보는 문의해주세요.\n"
        response += f"📞 고객센터: 1577-4321"
        
        return response
    
    def _get_product_response(self, query: str) -> str:
        """상품 정보 응답"""
        product_name = self._extract_product_name(query)
        
        if not product_name:
            return """📦 상품 정보 문의

찾으시는 상품명을 말씀해주세요.

💡 예시:
• '토끼 골드 정보'
• '은메달 상품'
• '2023 금바'

📞 고객센터: 1577-4321"""
        
        products = self._search_products(product_name)
        
        if not products:
            return f"""❌ '{product_name}' 상품을 찾을 수 없습니다.

💡 검색 팁:
• '토끼', '골드', '메달', '은', '금' 등 키워드로 검색
• 연도와 동물: '2023 토끼', '2024 용'

📞 직접 문의: 1577-4321"""
        
        response = f"📦 '{product_name}' 관련 상품 정보:\n\n"
        
        for i, product in enumerate(products[:3], 1):
            response += f"📦 {i}. {product['상품명']}\n"
            response += f"   💰 가격: {product['판매가']:,}원\n"
            response += f"   ⚖️  중량: {product['중량']}g\n"
            response += f"   🏷️  소재: {product['소재']}\n"
            response += f"   📐 규격: {product.get('규격', '정보 없음')}\n\n"
        
        response += f"💡 더 자세한 정보는 문의해주세요.\n"
        response += f"📞 고객센터: 1577-4321"
        
        return response
    
    def _extract_product_name(self, query: str) -> Optional[str]:
        """질문에서 상품명 추출"""
        # 연도 패턴
        year_pattern = r'(\d{4})\s*년'
        year_match = re.search(year_pattern, query)
        
        # 동물 키워드
        animals = ['토끼', '용', '뱀', '호랑이', '소', '말', '양', '원숭이', '닭', '개', '돼지']
        
        # 소재 키워드
        materials = ['골드', '메달', '은', '금', '카드형', '바', '코인']
        
        query_lower = query.lower()
        
        # 연도+동물+소재 조합
        if year_match:
            year = year_match.group(1)
            for animal in animals:
                if animal in query:
                    for material in materials:
                        if material in query:
                            return f"{year} {animal} {material}"
        
        # 간단 키워드 검색
        for keyword in ['골드', '메달', '은', '금', '토끼', '용', '뱀', '카드형']:
            if keyword in query:
                return keyword
        
        return None
    
    def _search_products(self, product_name: str) -> List[Dict]:
        """상품 검색 (캐시 사용)"""
        if product_name in self._cache:
            return self._cache[product_name]
        
        try:
            query = """
                SELECT 상품명, 중량, [ 판매가 ] as 판매가, 소재, 규격, 구성
                FROM products 
                WHERE 상품명 LIKE ? 
                AND [ 판매가 ] IS NOT NULL 
                AND CAST([ 판매가 ] AS INTEGER) > 0
                ORDER BY CAST([ 판매가 ] AS INTEGER) DESC
                LIMIT 10
            """
            
            df = pd.read_sql(query, self.conn, params=[f"%{product_name}%"])
            
            if len(df) > 0:
                results = df.to_dict('records')
                self._cache[product_name] = results
                return results
            
            return []
            
        except Exception as e:
            print(f"검색 오류: {e}")
            return []
    
    def _get_greeting_response(self) -> str:
        """인사 응답"""
        return """👋 안녕하세요! 상품 정보 챗봇입니다.

무엇을 도와드릴까요?

💡 질문 예시:
• '토끼 골드 가격'
• '은메달 정보'
• '2023 금바'

📞 고객센터: 1577-4321"""
    
    def _get_default_response(self, query: str) -> str:
        """기본 응답"""
        return f"""❓ '{query}'에 대한 응답을 찾을 수 없습니다.

💡 도움이 필요하시면:
• 상품명 + '가격' 또는 '정보'로 질문
• '토끼 골드 가격'처럼 구체적으로 질문

📞 직접 문의: 1577-4321
📧 이메일: support@subbot.com"""

# 테스트 함수
def test_direct_chatbot():
    bot = DirectProductChatBot()
    
    test_queries = [
        "2023년 금바 상품 정보를 알려주세요",
        "토끼 골드 가격이 얼마인가요?",
        "은메달 상품 정보",
        "안녕하세요",
        "카드형 골드",
        "2024 용 메달 가격"
    ]
    
    print("🤖 Direct Product ChatBot 테스트 (LLM 없음)\n" + "="*60)
    
    for i, query in enumerate(test_queries, 1):
        print(f"\n📝 테스트 {i}: {query}")
        print("-" * 50)
        
        response = bot.get_product_info(query)
        print(f"🤖 응답: {response}")
        
        print("-" * 50)
    
    print("\n" + "="*60)
    print("🏁 Direct ChatBot 테스트 완료")

# 실행
if __name__ == "__main__":
    test_direct_chatbot()

🤖 Direct Product ChatBot 테스트 (LLM 없음)

📝 테스트 1: 2023년 금바 상품 정보를 알려주세요
--------------------------------------------------
🤖 응답: 📦 '금' 관련 상품 정보:

📦 1. 천마총 금관 카드형 골드 37.5g
   💰 가격: 5,900,000원
   ⚖️  중량: 37.5g
   🏷️  소재: Au 999.9
   📐 규격: 골드바 22x38mm 카드 86x54mm

📦 2. 한국의 대표 화가 3차 - 김환기 기념메달 금메달(매화와 항아리)
   💰 가격: 5,400,000원
   ⚖️  중량: 31.1g
   🏷️  소재: Au 999
   📐 규격: 31.1g / Φ40mm / 프루프

📦 3. 한국의 대표 화가 3차 - 김환기 기념메달 금메달(영원의 노래)
   💰 가격: 5,400,000원
   ⚖️  중량: 31.1g
   🏷️  소재: Au 999
   📐 규격: 31.1g / Φ40mm / 프루프

💡 더 자세한 정보는 문의해주세요.
📞 고객센터: 1577-4321
--------------------------------------------------

📝 테스트 2: 토끼 골드 가격이 얼마인가요?
--------------------------------------------------
🤖 응답: 💰 '골드' 관련 상품 가격 정보:

📦 1. 2025 을사년 뱀의 해 카드형 골드 37.5g
   💵 가격: 5,960,000원
   ⚖️  중량: 37.5g
   🏷️  소재: Au 999.9

📦 2. 2023 계묘년 토끼의 해 카드형 골드 37.5g
   💵 가격: 5,900,000원
   ⚖️  중량: 37.5g
   🏷️  소재: Au 999.9

📦 3. 2024 갑진년 용의 해 카드형 골드 37.5g
   💵 가격: 5,900,000원
   ⚖️  중량: 37.5g
   🏷️  소재: Au 999.9

📦 4. 가화만사성 카드형 골드(37.5g

In [ ]:
# 웹에서 상품 정보 챗봇 API 테스트
import requests
import json

def test_product_chat_api():
    """웹 API 상품 정보 챗봇 테스트"""
    
    # API 엔드포인트
    url = "http://127.0.0.1:8000/api/product-chat"
    
    # 테스트 질문들
    test_queries = [
        "토끼 골드 가격",
        "은메달 상품 정보",
        "2023 금바 가격",
        "카드형 골드",
        "안녕하세요"
    ]
    
    print("🌐 웹 상품 정보 챗봇 API 테스트\n" + "="*60)
    
    for i, query in enumerate(test_queries, 1):
        print(f"\n📝 테스트 {i}: {query}")
        print("-" * 50)
        
        # API 요청
        data = {"message": query}
        
        try:
            response = requests.post(url, json=data, timeout=10)
            
            if response.status_code == 200:
                result = response.json()
                print(f"✅ 성공")
                print(f"📦 응답: {result['answer']}")
                print(f"📊 카테고리: {result['category']}")
                print(f"🎯 신뢰도: {result['confidence']}")
                print(f"🔍 데이터 소스: {result['source']}")
            else:
                print(f"❌ 오류: {response.status_code}")
                print(f"📄 응답: {response.text}")
                
        except requests.exceptions.ConnectionError:
            print("❌ 연결 오류 - 백엔드 서버가 실행 중인지 확인하세요")
            print("💡 실행 명령어: cd backend && python main.py")
        except requests.exceptions.Timeout:
            print("⏰ 타임아웃 오류 (10초)")
        except Exception as e:
            print(f"❌ 예외 오류: {str(e)}")
        
        print("-" * 50)
    
    print("\n" + "="*60)
    print("🏁 웹 API 테스트 완료")

# 웹 API 상태 확인
def check_api_status():
    """API 서버 상태 확인"""
    try:
        response = requests.get("http://127.0.0.1:8000/", timeout=5)
        if response.status_code == 200:
            print("✅ 백엔드 서버 정상 작동 중")
            return True
        else:
            print(f"❌ 백엔드 서버 응답 오류: {response.status_code}")
            return False
    except requests.exceptions.ConnectionError:
        print("❌ 백엔드 서버에 연결할 수 없습니다")
        print("💡 실행 명령어: cd backend && python main.py")
        return False
    except Exception as e:
        print(f"❌ 상태 확인 오류: {str(e)}")
        return False

# 실행
if __name__ == "__main__":
    print("🔍 API 서버 상태 확인...")
    if check_api_status():
        print("\n" + "="*60)
        print("🧪 웹 상품 정보 챗봇 API 테스트 시작...")
        test_product_chat_api()
    else:
        print("\n⚠️ 백엔드 서버를 먼저 시작해주세요")

In [1]:
# shop.db 경로 확인
import os

print("📁 현재 작업 디렉토리:", os.getcwd())
print("📁 data 폴더 내용:")
for item in os.listdir():
    if item.endswith('.db'):
        print(f"  - {item}")

print("\n📁 shop.db 절대 경로:")
shop_db_path = os.path.abspath("shop.db")
print(f"  {shop_db_path}")
print(f"  존재 여부: {os.path.exists(shop_db_path)}")

# 데이터베이스 연결 테스트
import sqlite3
conn = sqlite3.connect("shop.db")
cursor = conn.cursor()

# 상품 수 확인
cursor.execute("SELECT COUNT(*) FROM products")
product_count = cursor.fetchone()[0]
print(f"📦 총 상품 수: {product_count}")

# '토끼'가 포함된 상품 확인
cursor.execute("SELECT 상품명 FROM products WHERE 상품명 LIKE '%토끼%' LIMIT 5")
rabbit_products = cursor.fetchall()
print(f"🐰 '토끼'가 포함된 상품 ({len(rabbit_products)}개):")
for i, (name,) in enumerate(rabbit_products, 1):
    print(f"  {i}. {name}")

conn.close()

📁 현재 작업 디렉토리: c:\Users\user\SubBot-AI\data
📁 data 폴더 내용:
  - shop.db

📁 shop.db 절대 경로:
  c:\Users\user\SubBot-AI\data\shop.db
  존재 여부: True
📦 총 상품 수: 125
🐰 '토끼'가 포함된 상품 (5개):
  1. 2023 계묘년 토끼의 해 카드형 골드 1.87g
  2. 2023 계묘년 토끼의 해 카드형 골드 11.25g
  3. 2023 계묘년 토끼의 해 카드형 골드 3.75g
  4. 2023 계묘년 토끼의 해 카드형 골드 37.5g
  5. 디윰아트 2023 계묘년 토끼의 해 고심도 메달
